# Finalize POS tags and lemmas

Reproduces the finalization logic from `Merge_POS.ipynb` (which computes `final_pos`/`final_morphology`/`final_lemma` but never persists them) and writes a shareable `century, sentence_index, token_index, word, lemma, pos` table to `../Data/century_pos_lemma_final.csv`.

Combines three per-token annotation sources for the same corpus:
- `century_morphology_data_v6.csv` — SinMorphy (lemma candidates + POS candidates, `all_tags`/`all_morphologies` pipe-separated when ambiguous)
- `century_morphology_data_sinling_full.csv` — sinling POS tags
- `century_morphology_data_tnt_full.csv` — TnT tagger POS tags

The three files are row-aligned 1:1 (verified: identical `word`/`century`/`sentence_index` at every row index across all 64,813 rows), so they are combined positionally, exactly as `Merge_POS.ipynb` does.

In [1]:
import pandas as pd

DATA_DIR = "../Data"

In [2]:
sinmorphy_df = pd.read_csv(f"{DATA_DIR}/century_morphology_data_v6.csv")
sinling_df = pd.read_csv(f"{DATA_DIR}/century_morphology_data_sinling_full.csv")
tnt_df = pd.read_csv(f"{DATA_DIR}/century_morphology_data_tnt_full.csv")

len(sinmorphy_df), len(sinling_df), len(tnt_df)

(64813, 64813, 64813)

In [3]:
sinmorphy_df["tnt_pos"] = tnt_df["tnt_upd_pos"]
sinmorphy_df["sinling_pos"] = sinling_df["sinling_upd_pos"]

## Priority-vote finalization

Ported verbatim from `Merge_POS.ipynb`: when SinMorphy returns multiple candidate tags for a word, pick the candidate that matches TnT's tag, else the one that matches sinling's tag, else default to the first candidate.

In [4]:
def finalize_all_columns(row: pd.Series) -> pd.Series:
    all_tags_list = [t.strip() for t in str(row["all_tags"]).split("|")]
    all_morphs_list = [m.strip() for m in str(row["all_morphologies"]).split("|")]

    selected_index = 0

    if len(all_tags_list) > 1:
        tnt_res = row["tnt_pos"]
        sinling_res = row["sinling_pos"]

        if tnt_res != "UNKNOWN" and tnt_res in all_tags_list:
            selected_index = all_tags_list.index(tnt_res)
        elif sinling_res != "UNKNOWN" and sinling_res in all_tags_list:
            selected_index = all_tags_list.index(sinling_res)

    final_tag = all_tags_list[selected_index]

    try:
        final_morph = all_morphs_list[selected_index]
    except IndexError:
        final_morph = all_morphs_list[0] if all_morphs_list else "UNKNOWN"

    if "+" in final_morph:
        final_lemma = final_morph.split("+")[0]
    else:
        final_lemma = final_morph

    return pd.Series([final_tag, final_morph, final_lemma])

In [5]:
sinmorphy_df[["final_pos", "final_morphology", "final_lemma"]] = sinmorphy_df.apply(
    finalize_all_columns, axis=1
)

sinmorphy_df["final_pos"].value_counts()

final_pos
NOUN                      31890
Verb                       9963
PART                       9329
Proper noun or Invalid     9143
PROPER NOUN                2264
ADJECTIVE                  1661
PRONOUN                     546
FOREIGN WORD                 17
Name: count, dtype: int64

## Known issue: unparsed tokens

**Note:** ~14% of tokens (9,143 rows) have no SinMorphy analysis at all (`all_morphologies` is `NaN`, `all_tags` is the single value `"Proper noun or Invalid"`). For these, `finalize_all_columns` falls back to `str(row['all_morphologies']).split('|')`, which on a `NaN` produces the literal string `"nan"` as `final_lemma`.

That string survives fine in memory, but once written to CSV and reloaded (as any shared dataset will be), pandas' default `read_csv` treats the literal text `"nan"` as a null value — silently turning these into blank `lemma` cells. This does not show up if you only inspect `sinmorphy_df` in this notebook session; it only appears after the CSV round-trip needed to persist/share the file.

**A second, related edge case** (203 rows, e.g. the word `යයි`): some SinMorphy morphology candidates have no lemma stem at all before the first `+`, e.g. `+N+F+SG+DF+ACC+FN`, so `final_morph.split("+")[0]` yields an empty string rather than `NaN` — not caught by the check above, but the same underlying problem (a degenerate SinMorphy morphology string).

**Fix applied below:** for both cases, set `final_lemma = word` — matching the fallback the corpus already uses for `new_lemma` in `century_morphology_data_v6.csv` and in `Reconstruct_lemmas_OG.ipynb`. This only patches these two extraction edge cases; the priority-vote POS/lemma logic above is unchanged.

In [6]:
unparsed_mask = sinmorphy_df["all_morphologies"].isna()
empty_lemma_mask = sinmorphy_df["final_lemma"].astype(str).str.strip() == ""
fallback_mask = unparsed_mask | empty_lemma_mask
print(f"Unparsed tokens (no SinMorphy analysis): {unparsed_mask.sum()} / {len(sinmorphy_df)}")
print(f"Empty-stem morphology tokens: {empty_lemma_mask.sum()} / {len(sinmorphy_df)}")

sinmorphy_df.loc[fallback_mask, "final_lemma"] = sinmorphy_df.loc[fallback_mask, "word"]

Unparsed tokens (no SinMorphy analysis): 9143 / 64813
Empty-stem morphology tokens: 203 / 64813


## Assemble and save the finalized table

Adds `token_index` (position of the word within its `(century, sentence_index)` sentence) so downstream users can reconstruct sentence order after any resorting/filtering — the source files only encode order via row position, which a shared CSV cannot guarantee to preserve.

In [7]:
sinmorphy_df["token_index"] = sinmorphy_df.groupby(["century", "sentence_index"]).cumcount()

final_df = sinmorphy_df[
    ["century", "sentence_index", "token_index", "word", "final_lemma", "final_pos"]
].rename(columns={"final_lemma": "lemma", "final_pos": "pos"})

print("Remaining blank lemma cells:", final_df["lemma"].isna().sum())
final_df.head()

Remaining blank lemma cells: 0


,century,sentence_index,token_index,word,lemma,pos
0,5,0,0,කාම,කාම,NOUN
1,5,0,1,ලොක,ලො,NOUN
2,5,0,2,රූප,රූප,NOUN
3,5,0,3,ලොක,ලො,NOUN
4,5,0,4,අරූපලෝක,අරූපලෝ,NOUN


In [8]:
out_path = f"{DATA_DIR}/century_pos_lemma_final.csv"
final_df.to_csv(out_path, index=False, encoding="utf-8")
print(f"Wrote {len(final_df)} rows to {out_path}")

Wrote 64813 rows to ../Data/century_pos_lemma_final.csv
